# GroundedX / RAN-Doc revision runs (Kaggle, 2× T4)

**Settings → Accelerator: GPU T4 ×2, Internet: ON.** Then *Save Version → Save & Run All (Commit)*.

The notebook is resumable. Each session runs until `DEADLINE_HOURS`, then zips `results/v2` to
`/kaggle/working/results_v2.zip`. For the next session, add the previous version's output as an
input (*Add Input → Your Work → this notebook*) and run again. Finished seeds are kept and unfinished
ones continue.

Order: accuracy jobs (E1/E3 first, then the E2 sweep, then E7 k=1/3). Profiling runs after that
(`RUN_PROFILING`) and is tagged `t4`. Edge-GPU (RTX 3050) profiling is a separate local run.

In [ ]:
# ---- configuration ----
REPO_URL = "https://github.com/TanishqX-404/groundedx.git"
BRANCH = "revision-minds"
DEADLINE_HOURS = 11.0      # Kaggle hard limit is 12 h; leaves time to zip outputs
RUN_ACCURACY = True
RUN_PROFILING = True       # runs only after all accuracy jobs are finished
GPUS = ["0", "1"]

In [ ]:
import glob, os, subprocess, sys, time, zipfile, shutil
T0 = time.time()
!nvidia-smi
WORK = "/kaggle/working"
REPO = f"{WORK}/groundedx"
# Code: a Kaggle dataset containing the repo (if attached) wins, otherwise clone the branch.
src = [p for p in glob.glob("/kaggle/input/*/groundedx*") if os.path.isdir(p) and os.path.exists(f"{p}/pyproject.toml")]
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if src:
    shutil.copytree(src[0], REPO)
    print("code from dataset", src[0])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
# Restore results from a previous session (output of an earlier version attached as input).
for z in glob.glob("/kaggle/input/**/results_v2.zip", recursive=True):
    print("restoring", z)
    with zipfile.ZipFile(z) as f:
        f.extractall(f"{REPO}/results")
os.chdir(REPO)
!git log --oneline -1 || true

In [ ]:
# Build llama.cpp's llama-server with CUDA at the pinned build used for all revision runs
# (~10-15 min; static binary, T4 = sm_75). Skipped if a binary already exists.
LLAMA_TAG = "b11398"
BIN = f"{WORK}/llama-bin/llama-server"

def sh(cmd):
    print("+", cmd[:200], flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
    return r.returncode == 0

if not os.path.exists(BIN):
    nvcc = shutil.which("nvcc") or next(iter(glob.glob("/usr/local/cuda*/bin/nvcc")), None)
    assert nvcc, "nvcc not found - is the GPU accelerator enabled?"
    cuda_root = os.path.dirname(os.path.dirname(os.path.realpath(nvcc)))
    # libcuda: prefer the real driver library mounted by the GPU runtime, else the toolkit stub.
    libcuda = next(iter(
        glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*") + glob.glob("/usr/lib64/libcuda.so*")
        + glob.glob(f"{cuda_root}/lib64/stubs/libcuda.so") + glob.glob(f"{cuda_root}/targets/*/lib/stubs/libcuda.so")
    ), None)
    print("nvcc:", nvcc, "| cuda root:", cuda_root, "| libcuda:", libcuda)
    sh(f"rm -rf /tmp/llama.cpp && git -c advice.detachedHead=false clone -q --depth 1 -b {LLAMA_TAG} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    base = (f"cd /tmp/llama.cpp && rm -rf build && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
            f"-DCMAKE_CUDA_COMPILER={nvcc} -DCUDAToolkit_ROOT={cuda_root} "
            "-DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF")
    attempts = []
    if libcuda:
        attempts.append(("driver-lib", f"{base} -DCUDA_cuda_driver_LIBRARY={libcuda}"))
    # Fallback: legacy CUDA memory pool, which does not link libcuda directly.
    attempts.append(("no-vmm", f"{base} -DGGML_CUDA_NO_VMM=ON"))
    for name, cfg_cmd in attempts:
        if sh(cfg_cmd) and sh("cd /tmp/llama.cpp && cmake --build build --config Release -j4 --target llama-server"):
            print("build OK with", name)
            open(f"{WORK}/llama_build_variant.txt", "w").write(name)
            break
    else:
        raise RuntimeError("llama-server build failed with all configurations (see output above)")
    os.makedirs(f"{WORK}/llama-bin", exist_ok=True)
    shutil.copy("/tmp/llama.cpp/build/bin/llama-server", BIN)
os.environ["LLAMA_SERVER_BIN"] = BIN
print(subprocess.run([BIN, "--version"], capture_output=True, text=True).stderr)
!pip install -q -e . nvidia-ml-py xgboost huggingface_hub pytest

In [ ]:
import yaml
from huggingface_hub import hf_hub_download
cfg = yaml.safe_load(open("configs/experiment.yaml"))
os.makedirs("models", exist_ok=True)
for key, m in cfg["generation"]["models"].items():
    print(key, hf_hub_download(m["repo"], m["file"], local_dir="models"))
!ls -la models

In [ ]:
# Correctness gate (claims tests C1-C8) incl. a real constrained decode through llama-server on GPU.
!GROUNDEDX_TEST_MODEL=models/qwen2.5-1.5b-instruct-q4_k_m.gguf python -m pytest -q tests -x 2>&1 | tail -5

In [ ]:
# Speed probe: 10 windows of the main config (separate experiment name, not used in results).
!python scripts/run_llm.py --model qwen2.5-3b-q4 --model-path models/qwen2.5-3b-instruct-q4_k_m.gguf --mode rag --seeds 7 --limit 10 --experiment _speedprobe
import json, statistics
rows = [json.loads(l) for l in open("results/v2/_speedprobe/seed_7/predictions.jsonl")]
print("median s/call:", statistics.median(r["latency_ms"] for r in rows) / 1000, "valid:", sum(r["valid"] for r in rows), "/", len(rows))
shutil.rmtree("results/v2/_speedprobe")

In [ ]:
remaining = DEADLINE_HOURS - (time.time() - T0) / 3600
if RUN_ACCURACY:
    !python scripts/run_queue.py --phase accuracy --gpus {" ".join(GPUS)} --models-dir models --deadline-hours {remaining:.2f}

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("rq", "scripts/run_queue.py"); rq = importlib.util.module_from_spec(spec); spec.loader.exec_module(rq)
acc_left = [j for j in rq.ACCURACY_JOBS if not rq.accuracy_done(j)]
print("accuracy jobs left:", len(acc_left))
remaining = DEADLINE_HOURS - (time.time() - T0) / 3600
if RUN_PROFILING and not acc_left and remaining > 0.3:
    # one GPU at a time so the two profiles do not contend for CPU
    !python scripts/run_queue.py --phase profile --gpus 0 --models-dir models --tag t4 --deadline-hours {remaining:.2f}

In [ ]:
!python scripts/make_results.py
with zipfile.ZipFile(f"{WORK}/results_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob("results/v2/**", recursive=True):
        if os.path.isfile(path):
            z.write(path, os.path.relpath(path, "results"))
print("zipped ->", f"{WORK}/results_v2.zip", os.path.getsize(f"{WORK}/results_v2.zip") // 1024, "KB")
print(open("results/v2/RESULTS.md").read()[:6000])